In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [3]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [4]:
documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(index, openai_client)

In [ ]:
# assistant.rag('I just discovered the course, can I still join?')

'Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still open.'

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes — in most cases, you can still join a course after it has started, but it depends on the course’s enrollment policy and schedule.\n\nA few things to check:\n- **Enrollment deadline**: Some courses allow late registration.\n- **Class capacity**: There may be available seats.\n- **Prerequisites or access requirements**: You may need approval or prior coursework.\n- **Missed work**: You might need to catch up on lectures, assignments, or attendance.\n\nIf you want, I can help you draft a message to the instructor or registrar asking whether late enrollment is still possible.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [24]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [43]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)
response.output

[ResponseFunctionToolCall(arguments='{"query":"can I join the course late discovered the course enrollment join after start"}', call_id='call_D0vmPA9NP2DeNaSXlNCsvdU0', name='search', type='function_call', id='fc_062fa39bcbb6fb94006a324c2ab0bc8195abb81c9fdfbc5bd4', namespace=None, status='completed')]

In [44]:
print(json.dumps(response.model_dump(), indent=2))

{
  "id": "resp_062fa39bcbb6fb94006a324c29cbbc81958fc2e5064bcf4f4a",
  "created_at": 1781681193.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "arguments": "{\"query\":\"can I join the course late discovered the course enrollment join after start\"}",
      "call_id": "call_D0vmPA9NP2DeNaSXlNCsvdU0",
      "name": "search",
      "type": "function_call",
      "id": "fc_062fa39bcbb6fb94006a324c2ab0bc8195abb81c9fdfbc5bd4",
      "namespace": null,
      "status": "completed"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "search",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Search query text to look up in the course FAQ."
          }
        },
        "required": [
          "

In [46]:
for item in response.output:
    if item.type == "function_call":
        print("✅ Model decided to use tool:", item.name)
        print("   With arguments:", item.arguments)
    elif item.type == "message":
        print("❌ Model answered directly, no tool used")
        print("   Answer:", item.content[0].text)

✅ Model decided to use tool: search
   With arguments: {"query":"can I join the course late discovered the course enrollment join after start"}


In [47]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [25]:
response

Response(id='resp_030b1f2e09b539dc006a3247d17e648194a53fc5d0110e4d8f', created_at=1781680081.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late after it has started? discovered the course just now join it"}', call_id='call_jeLZTVZmRKEtOe6F0x3nF8I5', name='search', type='function_call', id='fc_030b1f2e09b539dc006a3247d2cae48194aa112c6f77f364c6', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')], top_p=0.98, background=False, complete

In [49]:
import json

call = response.output[0]
args = json.loads(call.arguments)

In [50]:
call

ResponseFunctionToolCall(arguments='{"query":"can I join the course late discovered the course enrollment join after start"}', call_id='call_D0vmPA9NP2DeNaSXlNCsvdU0', name='search', type='function_call', id='fc_062fa39bcbb6fb94006a324c2ab0bc8195abb81c9fdfbc5bd4', namespace=None, status='completed')

In [51]:
args

{'query': 'can I join the course late discovered the course enrollment join after start'}

In [52]:

results = search(**args)
result_json = json.dumps(results, indent=2)

In [53]:
print(result_json, '\n')

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"can I join the course late discovered the course enrollment join after start"}', call_id='call_D0vmPA9NP2DeNaSXlNCsvdU0', name='search', type='function_call', id='fc_062fa39bcbb6fb94006a324c2ab0bc8195abb81c9fdfbc5bd4', namespace=None, status='completed')]

In [ ]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [60]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"can I join the course late discovered the course enrollment join after start"}', call_id='call_D0vmPA9NP2DeNaSXlNCsvdU0', name='search', type='function_call', id='fc_062fa39bcbb6fb94006a324c2ab0bc8195abb81c9fdfbc5bd4', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_D0vmPA9NP2DeNaSXlNCsvdU0',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",

In [61]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)
response.output

[ResponseOutputMessage(id='msg_062fa39bcbb6fb94006a324ce05ef88195a7ed473386e8bd05', content=[ResponseOutputText(annotations=[], text='Yes — you can still join. The course materials are available, and you can start anytime.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [62]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [63]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [65]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        'type': 'function_call_output',
        'call_id': call.call_id,
        'output': result_json,
    }


In [66]:
question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question},
    ]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool],
    )

messages.extend(response.output)

has_function_call = False

In [73]:
for item in response.output:
    if item.type == 'function_call':
        print('function_call: ', item.name, '\n', item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_call = True
    elif item.type == 'message':
        print("ASSISTANT:")
        print(item.content[0].text)

function_call:  search 
 {"query":"join course discovered late enrollment can I join course FAQ"}


In [74]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still open. If you’re just learning for the content, you can start anytime.

If you want, I can also help you figure out how to start the course and what the weekly workflow looks like.


In [75]:
def agent_loop(instructions, question, model='gpt-5.4-mini'):
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question},
        ]
    
    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="gpt-5.4-mini",
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                print(item.content[0].text)
                last_answer = item.content[0].text

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [76]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local model Ollama FAQ"}
function_call: search {"query":"How to run Ollama locally course FAQ"}
function_call: search {"query":"local Ollama setup run model on your machine FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model**
   Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This will download the model and start it locally with a chat-style prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing Ollama is up.

4. **Use it from Python**
   Install the client:
   ```bash
   pip install ollama
   ```
   Example:
   ```python
   import ollama


'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - **macOS**: download and install the `.pkg`\n   - **Windows**: download and install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model**\n   Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start it locally with a chat-style prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing Ollama is up.\n\n4. **Use it from Python**\n   Install the client:\n   ```bash\n   pip install ollama\n   ```\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a “connection refused” error, restart the server with:\n```

In [77]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join late enrollment enrollment FAQ"}
iteration #2...
function_call: search {"query":"certificate project while still accepting submissions peer-review form closed live cohort self-paced FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. Certificates are only available for the live cohort, not for self-paced participation.

If you'd like, I can also help you figure out how to start the course or explain the certificate requirements in more detail.


"Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. Certificates are only available for the live cohort, not for self-paced participation.\n\nIf you'd like, I can also help you figure out how to start the course or explain the certificate requirements in more detail."

In [78]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess definition opening purpose decline accepted"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening moves d4 d5 c4 accepted declined meaning"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a gambit to try to **gain control of the center** and tempt Black’s d-pawn away from the center.

### Main idea
- White wants to build a strong center and get active piece play.
- Black can choose to:
  - **Accept** the gambit: `dxc4`
  - **Decline** the gambit: keep the pawn protected, often with `e6` or `c6`

### Why it’s called a “gambit”
A **gambit** means sacrificing a pawn early for a positional advantage, such as:
- better center control
- faster development
- more active pieces

### Common forms
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it
- **Sla

'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a gambit to try to **gain control of the center** and tempt Black’s d-pawn away from the center.\n\n### Main idea\n- White wants to build a strong center and get active piece play.\n- Black can choose to:\n  - **Accept** the gambit: `dxc4`\n  - **Decline** the gambit: keep the pawn protected, often with `e6` or `c6`\n\n### Why it’s called a “gambit”\nA **gambit** means sacrificing a pawn early for a positional advantage, such as:\n- better center control\n- faster development\n- more active pieces\n\n### Common forms\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n- **Slav Defense**: Black plays `c6`\n- **Semi-Slav**: `c6` and `e6` ideas combined\n\nIf you want, I can also show you the **typical move ideas** or a **simple beginner explanation** of how to play it.'